### Análisis de la temperatura

In [66]:
import os
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from RRAM import Representate as rp
from RRAM.Representate import config_ax, setup_paper_plt

In [67]:
def plot_temperature_profile(
    T_matrix: np.ndarray,
    atom_size: float,
    columna_x: int,
    step: int = 4,
    title: str = "Vertical Temperature Profile",
    save_path: str | None = None,
    usar_latex: bool = True,
    scaling: float = 2.5,
):
    """
    Genera un gráfico 2D del perfil de caída de temperatura vertical,
    empleando la configuración global de estilo para publicación.
    """
    # 1. Aplicar los estilos globales ANTES de crear la figura
    setup_paper_plt(plt, latex=usar_latex, scaling=scaling)

    # 2. Extraer los datos verticalmente (fijamos X, recorremos Y)
    data_y = T_matrix[:, columna_x]

    # 3. Calcular el vector de distancias físicas en nm (Eje X del plot)
    distancias = np.arange(0, T_matrix.shape[0]) * atom_size

    # 4. Crear la figura y los ejes
    fig, ax = plt.subplots(figsize=(12, 9))

    # 5. Aplicar la configuración de estilo específica para los ejes (Grid, ticks, etc.)
    config_ax(ax)

    # 6. Dibujar la línea continua con puntos espaciados según 'step'
    ax.plot(
        distancias,
        data_y,
        color="#1976D2",  # Azul técnico
        linewidth=2.5,
        label=f"Posición X = {columna_x}",
        marker="o",
        markersize=6 * (scaling / 2.5),  # Escala el punto con el tamaño global
        markevery=step,
        markerfacecolor="white",
        markeredgewidth=1.5,
    )

    # 7. Configurar etiquetas y estética usando comandos de siunitx
    ax.set_xlabel(r"Vertical Distance (\si{\nano\meter})")
    ax.set_ylabel(r"Temperature (K)")
    ax.set_title(title, pad=15)

    ax.legend(loc="upper right")

    plt.tight_layout()

    # 5. Guardar la gráfica
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

        # Opcional: guardar en pdf también
        # ruta_pdf = os.path.splitext(save_path)[0] + ".pdf"
        # plt.savefig(ruta_pdf, bbox_inches="tight")

    plt.close(fig)

    return None


In [68]:
def extraer_perfiles_temperatura(lista_matrices: list, etiquetas: list, columna_x: int, atom_size: float):
    """
    Extrae el perfil vertical de temperatura para múltiples matrices.

    Argumentos:
    - lista_matrices: Lista de arrays 2D de temperatura.
    - etiquetas: Nombres para la leyenda (ej: ["1.0 V", "1.5 V", "2.0 V"]).
    - columna_x: Índice de la columna a evaluar.
    - atom_size: Tamaño físico de cada celda en nm.

    Retorna:
    - distancias: Array 1D con el eje X físico.
    - perfiles: Diccionario con formato { "etiqueta": array_temperaturas_1D }
    """
    if len(lista_matrices) != len(etiquetas):
        raise ValueError("Debe haber el mismo número de matrices que de etiquetas.")

    # Usamos la forma de la primera matriz para calcular las distancias
    distancias = np.arange(0, lista_matrices[0].shape[0]) * atom_size

    perfiles = {}
    for matriz, etiqueta in zip(lista_matrices, etiquetas):
        # Extraemos solo la columna deseada
        perfiles[etiqueta] = matriz[:, columna_x]

    return distancias, perfiles


In [69]:
def plot_multiples_perfiles(
    distancias: np.ndarray,
    perfiles: dict,
    columna_x: int,
    step: int = 1,
    title: str = "Vertical Temperature Profile Evolution",
    save_path: str | None = None,
    usar_latex: bool = True,
    scaling: float = 2.5,
):
    """
    Dibuja múltiples perfiles de temperatura en una misma gráfica,
    manteniendo la estética exacta de línea y marcadores.
    """
    # 1. Aplicar los estilos globales ANTES de crear la figura
    setup_paper_plt(plt, latex=usar_latex, scaling=scaling)

    # 2. Crear la figura y los ejes
    fig, ax = plt.subplots(figsize=(12, 9))

    # 3. Aplicar la configuración de estilo específica para los ejes (Grid, ticks, etc.)
    config_ax(ax)

    # 4. Generar una paleta de colores dinámicos (para que se distingan las líneas)
    viridis = plt.get_cmap('viridis')  # Obtener el colormap 'viridis'
    colores = viridis(np.linspace(0, 0.9, len(perfiles)))

    # 5. Dibujar cada línea con la MISMA ESTÉTICA que tu función original
    for (etiqueta, data_y), color in zip(perfiles.items(), colores):
        ax.plot(
            distancias,
            data_y,
            color=color,  # El color cambia en cada iteración
            linewidth=2.5,
            label=etiqueta,  # La leyenda corresponde a cada voltaje o iteración
            marker="o",
            markersize=6 * (scaling / 2.5),
            markevery=step,
            markerfacecolor="white",  # Mantiene el interior del punto blanco
            markeredgewidth=1.5,  # Mantiene el grosor del borde
        )

    # 6. Configurar etiquetas y estética usando comandos de siunitx
    ax.set_xlabel(r"Vertical Distance (\si{\nano\meter})")
    ax.set_ylabel(r"Temperature (K)")
    ax.set_title(title, pad=15)

    # Añadir la leyenda para saber qué es cada línea
    ax.legend(loc="upper right")

    plt.tight_layout()

    # 7. Guardar la gráfica
    if save_path:
        os.makedirs(os.path.dirname(save_path), exist_ok=True)
        plt.savefig(save_path, dpi=300, bbox_inches="tight")

        # Opcional: guardar en pdf también
        # ruta_pdf = os.path.splitext(save_path)[0] + ".pdf"
        # plt.savefig(ruta_pdf, bbox_inches="tight")

    plt.close(fig)

    return None


In [70]:
atom_size = 0.25e-9
num_simulation = 2
voltage = 0.88
figures_path = f"C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Results/simulation_{num_simulation}/Figures/"
paso = 4
filename_temp = f"temperatura_{voltage}_pp_set"

# Cargar archivo .npz
temperatura = np.load(f"C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Results/simulation_{num_simulation}/Figures/" + filename_temp + ".npy")

# Ver  contiene
print(temperatura)


[[300.         317.45255139 346.96522539 ... 347.3939289  317.61174877
  300.        ]
 [300.         320.14876346 347.71938829 ... 348.15463094 320.33240514
  300.        ]
 [300.         320.17918681 347.79122934 ... 348.22641224 320.36279149
  300.        ]
 ...
 [300.         310.37677067 324.5806932  ... 324.76433753 310.45438899
  300.        ]
 [300.         310.36595689 324.55508527 ... 324.73802095 310.44327576
  300.        ]
 [300.         308.98090657 324.17261668 ... 324.35244514 309.04779981
  300.        ]]


In [71]:
plot_temperature_profile(
    T_matrix=temperatura,
    atom_size=atom_size,
    columna_x=50, 
    step=paso,  
    title=f"Vertical Temp. Drop ($V = {voltage}$ V)",
    save_path=os.path.join(figures_path, f"Temp_Profile_sim_{num_simulation}.png"),
)

In [ ]:
import numpy as np

# Parámetros que permanecen inalterados
atom_size = 0.25  # Nota: Lo pongo como 0.25 (nm) para que el eje X del plot salga directamente en nanómetros
voltage = 0.88
paso = 4
filename_temp = f"temperatura_{voltage}_pp_set.npy"
base_path = "C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Results/"

# Listas vacías donde guardaremos las matrices y los nombres para la leyenda

valores_manuales = [1.5e-06, 1.2e-06, 1.0e-06, 8.5e-07, 7.0e-07, 5.5e-07, 4.0e-07, 2.5e-07, 1.0e-07, 5.0e-08]

lista_matrices = []
etiquetas = []

# Bucle para cargar las 10 simulaciones (del 1 al 10 inclusive)
for num_simulation in range(1, 4):
    # --- LA MODIFICACIÓN ---
    # Si la simulación es la 6, se la salta y pasa a la 7
    if num_simulation == [6 , 7]:
        print("Saltando la simulación 6...")
        continue
    # -----------------------
    
    # 1. Construir la ruta exacta para esta simulación específica
    file_path = f"{base_path}simulation_{num_simulation}/Figures/{filename_temp}"

    try:
        # 2. Cargar la matriz de temperatura
        matriz_temp = np.load(file_path)

        # 3. Almacenar la matriz y su etiqueta
        lista_matrices.append(matriz_temp)
        etiquetas.append(f"Sim. {num_simulation}")
        
        valor_actual = valores_manuales[num_simulation - 1]

    except FileNotFoundError:
        print(f"Advertencia: No se encontró el archivo para la simulación {num_simulation}. Ruta: {file_path}")

print(f"Se han cargado correctamente {len(lista_matrices)} matrices de temperatura.")

# =====================================================================
# A partir de aquí, simplemente llamas a tus funciones para extraer y plotear
# =====================================================================

# Seleccionas la columna que te interesa evaluar (ejemplo: 50)
columna_evaluar = 50

# 1. Extraemos los perfiles (usando la función del mensaje anterior)
distancias, perfiles_extraidos = extraer_perfiles_temperatura(
    lista_matrices=lista_matrices, etiquetas=etiquetas, columna_x=columna_evaluar, atom_size=atom_size
)

# 2. Ploteamos todas juntas
plot_multiples_perfiles(
    distancias=distancias,
    perfiles=perfiles_extraidos,
    columna_x=columna_evaluar,
    step=paso,
    title=f"Vertical Temp. Profile ($V = {voltage}$ V)",
    save_path=f"{base_path}Comparativa_Temperaturas_V_{voltage}.png",
)


Se han cargado correctamente 3 matrices de temperatura.


In [ ]:
import numpy as np

# Parámetros que permanecen inalterados
atom_size = 0.25
voltage = 0.88
paso = 4
filename_temp = f"temperatura_{voltage}_pp_set.npy"
base_path = "C:/Users/Usuario/Documents/GitHub/RRAM_Simulation/Results/"

# --- NUEVO: Tu vector de valores manuales (debe tener 10 elementos, uno por cada simulación del 1 al 10) ---
valores_manuales = [1.5e-06, 1.2e-06, 1.0e-06, 8.5e-07, 7.0e-07, 5.5e-07, 4.0e-07, 2.5e-07, 1.0e-07, 5.0e-08]

lista_matrices = []
etiquetas = []

for num_simulation in range(1, 11):
    # Saltamos las simulaciones que den problemas (ejemplo: 3, 6 y 8)
    if num_simulation in [3, 6, 8]:
        print(f"Saltando la simulación {num_simulation}...")
        continue

    file_path = f"{base_path}simulation_{num_simulation}/Figures/{filename_temp}"

    try:
        matriz_temp = np.load(file_path)
        lista_matrices.append(matriz_temp)

        # --- NUEVO: Extraemos el valor correspondiente a esta simulación ---
        # Restamos 1 porque la simulación 1 corresponde a la posición 0 de la lista
        valor_actual = valores_manuales[num_simulation - 1]

        # Guardamos la etiqueta formateada (puedes añadirle unidades si quieres, ej: f"{valor_actual} A")
        etiquetas.append(f"{valor_actual:.1e}")

    except FileNotFoundError:
        print(f"Advertencia: No se encontró el archivo para la simulación {num_simulation}.")

print(f"Se han cargado correctamente {len(lista_matrices)} matrices de temperatura.")

# =====================================================================
# Extracción y Ploteo
# =====================================================================

columna_evaluar = 50

distancias, perfiles_extraidos = extraer_perfiles_temperatura(
    lista_matrices=lista_matrices, etiquetas=etiquetas, columna_x=columna_evaluar, atom_size=atom_size
)

plot_multiples_perfiles(
    distancias=distancias,
    perfiles=perfiles_extraidos,
    columna_x=columna_evaluar,
    step=paso,
    title=f"Vertical Temp. Profile ($V = {voltage}$ V)",
    save_path=f"{base_path}Comparativa_Temperaturas_V_{voltage}.png",
)
